In [6]:
import pandas as pd
from sklearn.linear_model import LassoCV, Lasso, LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, cross_validate

df = pd.read_csv('SeoulBikeData.csv')
df = df.drop(columns=df.select_dtypes(include=object))

X = df.drop(columns='Rented Bike Count')
Y = df['Rented Bike Count']

pipe = Pipeline([
      ('norm',StandardScaler()),
      ('lasso',LassoCV(cv=10, random_state=0))
])

pipe.fit(X, Y)

lamb_opm = pipe['lasso'].alpha_
print(lamb_opm)

6.070019230823885


In [7]:
model = Pipeline([
    ('norm',StandardScaler()),
    ('ridge',Lasso(alpha=lamb_opm))
])

cv = KFold(n_splits=10,shuffle=True,random_state=1)

results = cross_validate(model,X,Y,cv=cv,
                         scoring={'MSE':'neg_mean_squared_error',
                                  'RMSE':'neg_root_mean_squared_error',
                                  'MAE':'neg_mean_absolute_error',
                                  'R2':'r2'})

rmse = -results['test_RMSE']
r2 = results['test_R2']

print('Resultado via Lasso Regression:')
print(f'RMSE: {rmse.mean():.4f} ± {rmse.std():.4f}')
print(f'R²:   {r2.mean():.4f} ± {r2.std():.4f}')


### COMPARANDO COM LEAST SQUARES


model = Pipeline([
    ('OLS',LinearRegression())
])

cv = KFold(n_splits=10,shuffle=True,random_state=1)

results = cross_validate(model,X,Y,cv=cv,
                         scoring={'MSE':'neg_mean_squared_error',
                                  'RMSE':'neg_root_mean_squared_error',
                                  'MAE':'neg_mean_absolute_error',
                                  'R2':'r2'})

rmse = -results['test_RMSE']
r2 = results['test_R2']

print('\nResultado via Linear Regression:')
print(f'RMSE: {rmse.mean():.4f} ± {rmse.std():.4f}')
print(f'R²:   {r2.mean():.4f} ± {r2.std():.4f}')

Resultado via Lasso Regression:
RMSE: 469.9495 ± 15.9815
R²:   0.4685 ± 0.0156

Resultado via Linear Regression:
RMSE: 469.6398 ± 15.9569
R²:   0.4692 ± 0.0158
